# Notebook 3: Developing estat.municipal_code_download_csv()

Created: 2026-07-07

### The Task

This notebook reworks the task in Notebook 2 using the newer version 0.1.5 of estatjp. This newer version downloads results that contain more than 100,000-rows per query. Large queries no longer need to split into smaller ones.

Version 0.1.5 also uses the ```urllib3``` package for manipulating query strings. The awkward url manipulations in Notebook 2 are not used in this notebook

This task also takes advantage of an observation that [System of Social and Demographic Statistics A Population and Households](https://www.e-stat.go.jp/en/dbview?sid=0000020101) only retrieves data from census years even though the website appears to provide for non census years. In other words, one does not need to filter out non census years.

Run the following in the terminal to install this package with its current functions and constants.

```
pip install .
```

In [1]:
import os
def get_relative_data_path():
    datapath = os.path.join(os.path.split(os.environ['VIRTUAL_ENV'])[0], 'data')
    reltodata = os.path.relpath(datapath,os.path.curdir)
    return reltodata

get_relative_data_path()

'..\\data'

### Set up for coding

Do the following in a terminal and in the Python .venv environment.

```
pip install python-dotenv
dotenv set ESTAT_APP_ID your-app-id
pip install estatjp
```

### Get populations for all areas

Again, the populations aren't necessary for our task but we have to retrieve something to get the names and codes of the municipalities. 

In [2]:
import datetime
from estatjp import api
from estatjp import exceptions as xs
import pandas as pd

# For the municipal codes in English from https://www.e-stat.go.jp/en/dbview?sid=0000020101
codeapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# For the municipal codes in Japanese from https://www.e-stat.go.jp/dbview?sid=0000020101
codeapiurlja = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=&appId=&lang=J&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# Create empty DataFrame
columns = ['census_year','muni_code','muni_name_en','muni_name_ja','population']
df_result = pd.DataFrame(columns=columns)

# Get the codes with municipal names in English.
try:
    dfs_en = api.get_csv_data(codeapiurlen,description=datetime.datetime.now())
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

try:
    dfs_ja = api.get_csv_data(codeapiurlja,description=datetime.datetime.now())
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

main_en = dfs_en.get('Main')
main_en = main_en[["SURVEY YEAR","time_code","area_code","AREA","value"]]
main_ja = dfs_ja.get('Main')
main_ja = main_ja[["調査年","time_code","area_code","地域","value"]]

maindf = pd.merge(main_en,main_ja, on=["time_code","area_code","value"] )
df_result = maindf[["SURVEY YEAR","area_code","AREA","地域","value"]]
df_result.columns = columns

print(df_result)


      census_year muni_code              muni_name_en muni_name_ja population
0            1980     01100      Hokkaido Sapporo-shi      北海道 札幌市    1401757
1            1985     01100      Hokkaido Sapporo-shi      北海道 札幌市    1542979
2            1990     01100      Hokkaido Sapporo-shi      北海道 札幌市    1671742
3            1995     01100      Hokkaido Sapporo-shi      北海道 札幌市    1757025
4            2000     01100      Hokkaido Sapporo-shi      北海道 札幌市    1822368
...           ...       ...                       ...          ...        ...
25058        2000     47382  Okinawa-ken Yonaguni-cho     沖縄県 与那国町       1852
25059        2005     47382  Okinawa-ken Yonaguni-cho     沖縄県 与那国町       1796
25060        2010     47382  Okinawa-ken Yonaguni-cho     沖縄県 与那国町       1657
25061        2015     47382  Okinawa-ken Yonaguni-cho     沖縄県 与那国町       1843
25062        2020     47382  Okinawa-ken Yonaguni-cho     沖縄県 与那国町       1676

[25063 rows x 5 columns]


This code works. Incorporate it into ```estat.py```.

In [3]:
from openczjp import estat

url_en = estat.API_URLS.get('Population_en')
url_en

'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

Run the following in the terminal to install this package with its current functions and constants.

```
pip install .
```

And test. Be sure to restart the Jupyter kernell and clear all outputs before testing to ensure that old code isn't interfering.

In [4]:
from openczjp import estat
estat.municipal_code_download_csv()

,census_year,muni_code,muni_name_en,muni_name_ja,population
0,1980,01100,Hokkaido Sapporo-shi,北海道 札幌市,1401757
1,1985,01100,Hokkaido Sapporo-shi,北海道 札幌市,1542979
2,1990,01100,Hokkaido Sapporo-shi,北海道 札幌市,1671742
3,1995,01100,Hokkaido Sapporo-shi,北海道 札幌市,1757025
4,2000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1822368
...,...,...,...,...,...
25058,2000,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1852
25059,2005,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1796
25060,2010,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1657
25061,2015,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1843
